Dataset: https://www.kaggle.com/datasets/yujikomi/feedbackprize-full-essays  
↑ Please vote.

Please comment if you have any problems.

In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
def get_essay(essay_id, DIR):
    essay_path = os.path.join(DIR, f"{essay_id}.txt")
    essay_text = open(essay_path, 'r').read()
    return essay_text

#https://www.kaggle.com/competitions/feedback-prize-2021/discussion/313330

from text_unidecode import unidecode
from typing import Dict, List, Tuple
import codecs
def replace_encoding_with_utf8(error: UnicodeError) -> Tuple[bytes, int]:
    return error.object[error.start : error.end].encode("utf-8"), error.end
def replace_decoding_with_cp1252(error: UnicodeError) -> Tuple[str, int]:
    return error.object[error.start : error.end].decode("cp1252"), error.end
codecs.register_error("replace_encoding_with_utf8", replace_encoding_with_utf8)
codecs.register_error("replace_decoding_with_cp1252", replace_decoding_with_cp1252)
def resolve_encodings_and_normalize(text: str) -> str:
    """Resolve the encoding problems and normalize the abnormal characters."""
    text = (
        text.encode("raw_unicode_escape")
        .decode("utf-8", errors="replace_decoding_with_cp1252")
        .encode("cp1252", errors="replace_encoding_with_utf8")
        .decode("utf-8", errors="replace_decoding_with_cp1252")
    )
    text = unidecode(text)
    return text

In [ ]:
train_1 = pd.read_csv('../input/feedback-prize-2021/train.csv')
train_2 = pd.read_csv('../input/feedback-prize-effectiveness/train.csv')
train_3 = pd.read_csv('../input/feedback-prize-english-language-learning/train.csv')

In [ ]:
train_1.head(2)

In [ ]:
train_2.head(2)

In [ ]:
train_3.head(2)

# Check essay inclusion

In [ ]:
essay_1 = train_1.id.unique()
print('Feedback 1: ' ,len(essay_1))

essay_2 = train_2.essay_id.unique()
print('Feedback 2: ', len(essay_2))

essay_3 = train_3.text_id.unique()
print('Feedback 3: ', len(essay_3))

In [ ]:
set(essay_2) - set(essay_1)

Feedback2 essay is included in Feedback1 essay.

In [ ]:
len(set(essay_3) - set(essay_1))

Of the essays in Feedback3, about 450 are included in Feedback1, while 3459 or so are new essays.

# Make all datasets

In [ ]:
essay_id = set(essay_1) - set(essay_3)

In [ ]:
train_1 = train_1[['id']].rename(columns={'id':'text_id'}).drop_duplicates()
train_1 = train_1[train_1.text_id.isin(essay_id)]
train_1['full_text'] = train_1['text_id'].apply(lambda x: get_essay(x, '../input/feedback-prize-2021/train/'))

In [ ]:
all_data = pd.concat([train_1, train_3]).reset_index(drop=True)
print(all_data.shape)
all_data.head()

a note of affiliation.

In [ ]:
all_data['in_fb1'] = all_data.text_id.apply(lambda x: x in essay_1)
all_data['in_fb2'] = all_data.text_id.apply(lambda x: x in essay_2)
all_data['in_fb3'] = all_data.text_id.apply(lambda x: x in essay_3)

In [ ]:
all_data['full_text'] = all_data['full_text'].apply(lambda x : resolve_encodings_and_normalize(x))

In [ ]:
all_data

In [ ]:
all_data.to_csv('./feedbackprize_full_essays.csv', index=False)

In [ ]:
all_data.info()